## Paths

In [1]:
from pathlib import Path
import pandas as pd
from collections import Counter
from typing import Optional
import re
import json


REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = REPO_ROOT / "data"
TRAIN_SDRF_DIR = DATA_DIR / "TrainingSDRFs"
TRAIN_SDRF_DIR.exists(), TRAIN_SDRF_DIR

(True, PosixPath('/Users/amysorekar1/kaggle_competition/data/TrainingSDRFs'))

## Inspecting highest filled columns to see term frequency

In [2]:
COLUMNS_TO_INSPECT = [
    "SourceName",
    "Organism", # ideal
    "AssayName",
    "comment[data file]",
    "Label",
    "Instrument", 
    "CleavageAgent",
    "FractionIdentifier",
    "Modification",
    "comment[file uri]",
    "TechnicalReplicate",
    "BiologicalReplicate",
    "FragmentMassTolerance",
    "PrecursorMassTolerance",
    "Disease",
    "Modification.1",
    "OrganismPart",
    "Sex",
    "technology type",
    "Age",
]

def inspect_column_values(sdrf_dir: Path, column: str, top_k: int = 20):
    if not sdrf_dir.exists():
        raise ValueError(f"Directory does not exist: {sdrf_dir}")

    counter = Counter()

    for tsv_path in sdrf_dir.glob("*.tsv"):
        try:
            df = pd.read_csv(tsv_path, sep="\t")
        except Exception as e:
            print(f"Failed to read {tsv_path}: {e}")
            continue

        if column not in df.columns:
            continue

        values = (
            df[column]
            .dropna()
            .astype(str)
            .str.strip()
        )

        values = values[values != ""]
        counter.update(values)

    return counter.most_common(top_k)

    
for col in COLUMNS_TO_INSPECT:
    print("\n" + "#" * 80)
    print(f"Column: {col}")
    print("#" * 80)

    values = inspect_column_values(TRAIN_SDRF_DIR, col, top_k=20)

    for val, count in values:
        print(f"{val:40s} {count}")


################################################################################
Column: SourceName
################################################################################
Proteometools                            1460
Sample 1                                 1094
Sample 2                                 961
Sample 3                                 940
Sample 5                                 923
Sample 4                                 920
Sample 6                                 917
Sample 7                                 839
Super SILAC Reference                    764
Sample 8                                 666
Sample 10                                627
Sample 9                                 618
V287                                     360
PXD012131-Sample-1                       326
Sample 11                                318
Sample 12                                289
Sample 13                                264
Sample 14                                262
Sample

# Vocabularies 

## Organism

In [3]:
ORGANISM_CANONICAL = {
    "Homo sapiens": [
        "homo sapiens",
        "human",
        "human samples",
        "human plasma",
        "h. sapiens",
    ],
    "Mus musculus": [
        "mus musculus",
        "mouse",
        "mice",
        "murine",
    ],
    "Escherichia coli": [
        "escherichia coli",
        "e. coli",
    ],
    "Escherichia coli K-12": [
        "escherichia coli k-12",
        "e. coli k-12",
    ],
    "Saccharomyces cerevisiae": [
        "saccharomyces cerevisiae",
        "yeast",
    ],
    "plasmodium falciparum": [
        "plasmodium falciparum",
        "p. falciparum",
        "plasmodium",
    ],
    "feces metagenome": [
        "fecal microbiome",
        "feces metagenome",
        "gut microbiome",
    ],
}

## Instrument 

In [36]:
INSTRUMENT_CANONICAL = {
    "Q Exactive HF": ["q exactive hf"],
    "Q Exactive Plus": ["q exactive plus"],
    "Q Exactive": ["q exactive"],
    "Orbitrap Fusion Lumos": ["orbitrap fusion lumos"],
    "Orbitrap Fusion": ["orbitrap fusion"],
    "LTQ Orbitrap Velos": ["ltq orbitrap velos"],
    "LTQ Orbitrap Elite": ["ltq orbitrap elite"],
    "LTQ Orbitrap XL": ["ltq orbitrap xl"],
}

## Cleavage Agent

In [5]:
CLEAVAGE_AGENT_CANONICAL = {
    "Trypsin": [
        "trypsin",
    ],
    "Trypsin/P": [
        "trypsin/p",
        "trypsin p",
    ],
    "Lys-C": [
        "lys-c",
        "lys c",
    ],
    "Chymotrypsin": [
        "chymotrypsin",
    ],
    "Asp-N": [
        "asp-n",
        "asp n",
    ],
    "Arg-C": [
        "arg-c",
        "arg c",
    ],
    "V8-DE": [
        "v8-de",
        "v8 de",
        "v8 protease",
    ],
    "Glutamyl endopeptidase": [
        "glutamyl endopeptidase",
    ],
    "leukocyte elastase": [
        "leukocyte elastase",
        "elastase",
    ],
    "unspecific cleavage": [
        "unspecific cleavage",
        "unspecific protease",
    ],
}

## Disease

In [6]:
DISEASE_CONTROL_CANONICAL = {
    "normal": [
        "normal",
        "healthy",
    ],
    "Control": [
        "control",
        "control sample",
        "control group",
    ],
    "uninfected": [
        "uninfected",
    ],
}

DISEASE_CANCER_KEYWORDS = [
    "melanoma",
    "adenocarcinoma",
    "carcinoma",
    "cancer",
]

DISEASE_SINGLETONS = {
    "Alzheimer's disease": [
        "alzheimer",
        "alzheimer's disease",
    ],
    "obesity": [
        "obesity",
        "obese",
    ],
}

DISEASE_SKIP_TERMS = [
    "trauma",
    "trauma victim",
    "dilated",
    "dilated left atria",
]


# Attribute detection

## Organism

In [7]:

def normalize_text(text: str) -> str:
    return re.sub(r"\s+", " ", text.lower())


def flatten_pubtext(pub_json: dict) -> str:
    parts: list[str] = []
    for v in pub_json.values():
        if isinstance(v, str):
            parts.append(v)
        elif isinstance(v, list):
            parts.extend(x for x in v if isinstance(x, str))
    return "\n".join(parts)


def is_reagent_context(text: str, trigger: str, window: int = 40) -> bool:
    for match in re.finditer(re.escape(trigger), text):
        start = max(0, match.start() - window)
        end = min(len(text), match.end() + window)
        context = text[start:end]
        if any(word in context for word in REAGENT_WORDS):
            return True
    return False


def choose_primary_organism(hits: list[str]) -> str:
    if len(hits) == 1:
        return hits[0]
    non_human = [h for h in hits if h != "Homo sapiens"]
    return non_human[0] if non_human else "Homo sapiens"


def extract_organism(
    paper_text: str,
    organism_canonical: dict[str, list[str]],
) -> Optional[str]:
    if not isinstance(paper_text, str) or not paper_text:
        return None

    text = normalize_text(paper_text)

    if any(sig in text for sig in HUMAN_STRONG_SIGNALS):
        return "Homo sapiens"

    hits: list[str] = []
    for canonical, triggers in organism_canonical.items():
        for trigger in triggers:
            if trigger in text and not is_reagent_context(text, trigger):
                hits.append(canonical)
                break

    return choose_primary_organism(hits) if hits else None


def extract_organism_from_paper(
    paper_path: Path,
    organism_canonical: dict[str, list[str]],
) -> Optional[str]:
    with open(paper_path) as f:
        paper_json = json.load(f)
    return extract_organism(flatten_pubtext(paper_json), organism_canonical)


def clean_str(x):
    return x.strip().lower().replace(" ", "") if isinstance(x, str) else x


def evaluate_predictions(
    pubtext_dir: Path,
    sdrf_dir: Path,
    organism_canonical: dict[str, list[str]],
) -> pd.DataFrame:
    rows = []

    for json_file in pubtext_dir.glob("*.json"):
        pxd = json_file.stem.split("_")[0]
        pred = extract_organism_from_paper(json_file, organism_canonical)

        sdrf_path = sdrf_dir / f"{pxd}_cleaned.sdrf.tsv"
        if sdrf_path.exists():
            gold_df = pd.read_csv(sdrf_path, sep="\t")
            gold = "; ".join(gold_df["Organism"].unique())
        else:
            gold = "SDRF not found"

        rows.append({"Predicted": pred, "Gold": gold})

    return pd.DataFrame(rows)


def get_mismatches(df: pd.DataFrame) -> pd.DataFrame:
    cleaned = df.copy()
    cleaned["Predicted_clean"] = cleaned["Predicted"].apply(clean_str)
    cleaned["Gold_clean"] = cleaned["Gold"].apply(clean_str)
    return cleaned[
        cleaned["Predicted_clean"] != cleaned["Gold_clean"]
    ][["Predicted", "Gold"]]

HUMAN_STRONG_SIGNALS = {
    "patients",
    "informed consent",
    "ethical approval",
    "helsinki",
    "surgery",
    "hospital",
    "aged ",
    "years old",
}

REAGENT_WORDS = {
    "antibody",
    "antibodies",
    "igg",
    "monoclonal",
    "polyclonal",
    "secondary",
    "primary antibody",
}


pubtext_dir = Path("data/TrainingPubText")
sdrf_dir = Path("data/TrainingSDRFs")

comparison_df = evaluate_predictions(
    pubtext_dir,
    sdrf_dir,
    ORGANISM_CANONICAL,
)

mismatches = get_mismatches(comparison_df)

print(f"Errors: {len(mismatches)}/{len(comparison_df)}")
print(f"Error rate: {100 * len(mismatches) / len(comparison_df):.2f}%")

Errors: 26/104
Error rate: 25.00%


## Instrument

In [43]:
def normalize_text(text: str) -> str:
    return re.sub(r"\s+", " ", text.lower())


def flatten_pubtext(pub_json: dict) -> str:
    parts = []
    for v in pub_json.values():
        if isinstance(v, str):
            parts.append(v)
        elif isinstance(v, list):
            parts.extend(x for x in v if isinstance(x, str))
    return "\n".join(parts)


def choose_primary_instrument(hits: list[str]) -> str:
    return hits[0]


def extract_instrument(
    paper_text: str,
    instrument_canonical: dict[str, list[str]],
) -> Optional[str]:
    if not isinstance(paper_text, str) or not paper_text:
        return None

    text = normalize_text(paper_text)
    hits = []

    for canonical, triggers in instrument_canonical.items():
        for trigger in triggers:
            if trigger in text:
                hits.append(canonical)
                break

    return choose_primary_instrument(hits) if hits else None


def extract_instrument_from_paper(
    paper_path: Path,
    instrument_canonical: dict[str, list[str]],
) -> Optional[str]:
    with open(paper_path) as f:
        paper_json = json.load(f)
    return extract_instrument(flatten_pubtext(paper_json), instrument_canonical)


def extract_gold_instrument(gold: str) -> Optional[str]:
    if not isinstance(gold, str):
        return None
    m = re.search(r"NT\s*=\s*([^;]+)", gold, flags=re.IGNORECASE)
    return m.group(1).strip() if m else gold.strip()


def evaluate_predictions(
    pubtext_dir: Path,
    sdrf_dir: Path,
    instrument_canonical: dict[str, list[str]],
) -> pd.DataFrame:
    rows = []

    for json_file in pubtext_dir.glob("*.json"):
        pxd = json_file.stem.split("_")[0]
        pred = extract_instrument_from_paper(json_file, instrument_canonical)

        sdrf_path = sdrf_dir / f"{pxd}_cleaned.sdrf.tsv"
        if sdrf_path.exists():
            gold_df = pd.read_csv(sdrf_path, sep="\t")
            gold_raw = "; ".join(gold_df["Instrument"].unique())
            gold = extract_gold_instrument(gold_raw)
        else:
            gold = None

        rows.append({"Predicted": pred, "Gold": gold})

    return pd.DataFrame(rows)


def get_mismatches(df: pd.DataFrame) -> pd.DataFrame:
    def match(pred, gold):
        if pred is None and gold is None:
            return True
        if pred is None or gold is None:
            return False
        return pred.lower() in gold.lower()

    mask = ~df.apply(lambda r: match(r["Predicted"], r["Gold"]), axis=1)
    return df[mask][["Predicted", "Gold"]]


pubtext_dir = Path("data/TrainingPubText")
sdrf_dir = Path("data/TrainingSDRFs")

comparison_df = evaluate_predictions(
    pubtext_dir,
    sdrf_dir,
    INSTRUMENT_CANONICAL,
)

# mismatches = get_mismatches(comparison_df)

print(f"Errors: {len(mismatches)}/{len(comparison_df)}")
print(f"Error rate: {100 * len(mismatches) / len(comparison_df):.2f}%")

# display(mismatches)

Errors: 71/104
Error rate: 68.27%


## Cleavage Agent

In [54]:
def normalize_text(text: str) -> str:
    return re.sub(r"\s+", " ", text.lower())


def flatten_pubtext(pub_json: dict) -> str:
    parts: list[str] = []
    for v in pub_json.values():
        if isinstance(v, str):
            parts.append(v)
        elif isinstance(v, list):
            parts.extend(x for x in v if isinstance(x, str))
    return "\n".join(parts)


def is_reagent_context(text: str, trigger: str, window: int = 40) -> bool:
    for match in re.finditer(re.escape(trigger), text):
        start = max(0, match.start() - window)
        end = min(len(text), match.end() + window)
        context = text[start:end]
        if any(word in context for word in REAGENT_WORDS):
            return True
    return False


def choose_primary_cleavage_agent(hits: list[str]) -> str:
    for c in CLEAVAGE_PRIORITY:
        if c in hits:
            return c
    return hits[0]


def extract_cleavage_agent(
    paper_text: str,
    cleavage_agent_canonical: dict[str, list[str]],
) -> Optional[str]:
    if not isinstance(paper_text, str) or not paper_text:
        return None

    text = normalize_text(paper_text)
    hits: list[str] = []

    for canonical, triggers in cleavage_agent_canonical.items():
        for trigger in triggers:
            if trigger in text and not is_reagent_context(text, trigger):
                hits.append(canonical)
                break

    if not hits:
        return "Trypsin"

    return choose_primary_cleavage_agent(hits)


def extract_cleavage_agent_from_paper(
    paper_path: Path,
    cleavage_agent_canonical: dict[str, list[str]],
) -> Optional[str]:
    with open(paper_path) as f:
        paper_json = json.load(f)
    return extract_cleavage_agent(
        flatten_pubtext(paper_json),
        cleavage_agent_canonical,
    )


def clean_str(x):
    return x.strip().lower().replace(" ", "") if isinstance(x, str) else x


def extract_nt_value(s: str) -> Optional[str]:
    if not isinstance(s, str):
        return None
    for part in s.split(";"):
        part = part.strip()
        if part.lower().startswith("nt="):
            return part[3:].strip()
    return s.strip()


def evaluate_predictions(
    pubtext_dir: Path,
    sdrf_dir: Path,
    cleavage_agent_canonical: dict[str, list[str]],
) -> pd.DataFrame:
    rows = []

    for json_file in pubtext_dir.glob("*.json"):
        pxd = json_file.stem.split("_")[0]
        pred = extract_cleavage_agent_from_paper(
            json_file,
            cleavage_agent_canonical,
        )

        sdrf_path = sdrf_dir / f"{pxd}_cleaned.sdrf.tsv"
        if sdrf_path.exists():
            gold_df = pd.read_csv(sdrf_path, sep="\t")
            gold = "; ".join(gold_df["CleavageAgent"].unique())
        else:
            gold = None

        rows.append({"Predicted": pred, "Gold": gold})

    return pd.DataFrame(rows)


def get_mismatches(df: pd.DataFrame) -> pd.DataFrame:
    cleaned = df.copy()

    cleaned["Predicted_norm"] = cleaned["Predicted"].apply(extract_nt_value)
    cleaned["Gold_norm"] = cleaned["Gold"].apply(extract_nt_value)

    cleaned["Predicted_clean"] = cleaned["Predicted_norm"].apply(clean_str)
    cleaned["Gold_clean"] = cleaned["Gold_norm"].apply(clean_str)

    return cleaned[
        cleaned["Predicted_clean"] != cleaned["Gold_clean"]
    ][["Predicted", "Gold"]]


CLEAVAGE_PRIORITY = [
    "Trypsin/P",
    "Lys-C",
    "Chymotrypsin",
    "Arg-C",
    "Asp-N",
    "V8-DE",
    "Glutamyl endopeptidase",
    "leukocyte elastase",
    "unspecific cleavage",
    "Trypsin",
]


pubtext_dir = Path("data/TrainingPubText")
sdrf_dir = Path("data/TrainingSDRFs")

comparison_df = evaluate_predictions(
    pubtext_dir,
    sdrf_dir,
    CLEAVAGE_AGENT_CANONICAL,
)

mismatches = get_mismatches(comparison_df)

print(f"Errors: {len(mismatches)}/{len(comparison_df)}")
print(f"Error rate: {100 * len(mismatches) / len(comparison_df):.2f}%")


Errors: 30/104
Error rate: 28.85%


## Disease

In [59]:
def normalize_text(text: str) -> str:
    return re.sub(r"\s+", " ", text.lower())


def flatten_pubtext(pub_json: dict) -> str:
    parts = []
    for v in pub_json.values():
        if isinstance(v, str):
            parts.append(v)
        elif isinstance(v, list):
            parts.extend(x for x in v if isinstance(x, str))
    return "\n".join(parts)


def extract_disease(
    paper_text: str,
    disease_singletons: dict[str, list[str]],
    disease_cancer_keywords: list[str],
    disease_skip_terms: list[str],
) -> Optional[str]:
    if not isinstance(paper_text, str) or not paper_text:
        return None

    text = normalize_text(paper_text)

    for skip in disease_skip_terms:
        if skip in text:
            return None

    for keyword in disease_cancer_keywords:
        if keyword in text:
            return keyword

    for canonical, triggers in disease_singletons.items():
        for trigger in triggers:
            if trigger in text:
                return canonical

    return None


def extract_disease_from_paper(
    paper_path: Path,
    disease_singletons: dict[str, list[str]],
    disease_cancer_keywords: list[str],
    disease_skip_terms: list[str],
) -> Optional[str]:
    with open(paper_path) as f:
        paper_json = json.load(f)
    return extract_disease(
        flatten_pubtext(paper_json),
        disease_singletons,
        disease_cancer_keywords,
        disease_skip_terms,
    )


def get_disease_column(df: pd.DataFrame) -> Optional[str]:
    for col in df.columns:
        if col.lower().endswith("disease"):
            return col
    return None


def clean_str(x):
    return x.strip().lower().replace(" ", "") if isinstance(x, str) else x


def evaluate_predictions(
    pubtext_dir: Path,
    sdrf_dir: Path,
    disease_singletons: dict[str, list[str]],
    disease_cancer_keywords: list[str],
    disease_skip_terms: list[str],
) -> pd.DataFrame:
    rows = []

    for json_file in pubtext_dir.glob("*.json"):
        pxd = json_file.stem.split("_")[0]
        pred = extract_disease_from_paper(
            json_file,
            disease_singletons,
            disease_cancer_keywords,
            disease_skip_terms,
        )

        sdrf_path = sdrf_dir / f"{pxd}_cleaned.sdrf.tsv"
        if sdrf_path.exists():
            gold_df = pd.read_csv(sdrf_path, sep="\t")
            disease_col = get_disease_column(gold_df)
            gold = (
                "; ".join(gold_df[disease_col].dropna().unique())
                if disease_col
                else None
            )
        else:
            gold = None

        rows.append({"Predicted": pred, "Gold": gold})

    return pd.DataFrame(rows)


def get_mismatches(df: pd.DataFrame) -> pd.DataFrame:
    cleaned = df.copy()
    cleaned["Predicted_clean"] = cleaned["Predicted"].apply(clean_str)
    cleaned["Gold_clean"] = cleaned["Gold"].apply(clean_str)
    return cleaned[
        cleaned["Predicted_clean"] != cleaned["Gold_clean"]
    ][["Predicted", "Gold"]]

pubtext_dir = Path("data/TrainingPubText")
sdrf_dir = Path("data/TrainingSDRFs")

comparison_df = evaluate_predictions(
    pubtext_dir,
    sdrf_dir,
    DISEASE_CONTROL_CANONICAL,
    DISEASE_SINGLETONS,
    DISEASE_CANCER_KEYWORDS,
)

mismatches = get_mismatches(comparison_df)

print(f"Errors: {len(mismatches)}/{len(comparison_df)}")
print(f"Error rate: {100 * len(mismatches) / len(comparison_df):.2f}%")

mismatches.head(50)

Errors: 103/104
Error rate: 99.04%


,Predicted,Gold
0,normal,dilated left atria
1,obesity,obesity; uninfected
2,None,not applicable; Non-small cell lung carcinoma
3,normal,None
4,normal,native_1; native_2; native_3; native_4; native...
6,None,normal
7,None,malignant melanoma; amelanotic melanoma
8,None,Triple-negative breast cancer
9,None,metastatic cutaneous melanoma
10,None,not available


In [61]:
import pandas as pd
from pathlib import Path

# paths
SAMPLE_SUB_PATH = Path("data/SampleSubmission.csv")
TEST_PUBTEXT_DIR = Path("data/TestPubText")

# load template
submission = pd.read_csv(SAMPLE_SUB_PATH)

# sanity check
assert submission.shape[1] == 81

# helper: map PXD -> extracted values
def build_lookup(extractor_fn):
    lookup = {}
    for json_file in TEST_PUBTEXT_DIR.glob("*.json"):
        pxd = json_file.stem.split("_")[0]
        lookup[pxd] = extractor_fn(json_file)
    return lookup


# build lookups using YOUR existing extractors
organism_lookup = build_lookup(
    lambda p: extract_organism_from_paper(p, ORGANISM_CANONICAL)
)

instrument_lookup = build_lookup(
    lambda p: extract_instrument_from_paper(p, INSTRUMENT_CANONICAL)
)

cleavage_lookup = build_lookup(
    lambda p: extract_cleavage_agent_from_paper(p, CLEAVAGE_AGENT_CANONICAL)
)

disease_lookup = build_lookup(
    lambda p: extract_disease_from_paper(
        p,
        DISEASE_CONTROL_CANONICAL,
        DISEASE_SINGLETONS,
        DISEASE_CANCER_KEYWORDS,
    )
)

# fill columns
def fill_column(col_name, lookup):
    submission[col_name] = submission["PXD"].map(lookup)
    submission[col_name] = submission[col_name].fillna("Not Applicable")


fill_column("Characteristics[Organism]", organism_lookup)
fill_column("Comment[Instrument]", instrument_lookup)
fill_column("Characteristics[CleavageAgent]", cleavage_lookup)
fill_column("Characteristics[Disease]", disease_lookup)

# everything else → Not Applicable (except ID, PXD, Raw Data File, Usage)
for col in submission.columns:
    if col in {
        "ID",
        "PXD",
        "Raw Data File",
        "Usage",
        "Characteristics[Organism]",
        "Comment[Instrument]",
        "Characteristics[CleavageAgent]",
        "Characteristics[Disease]",
    }:
        continue
    submission[col] = submission[col].fillna("Not Applicable")

# final checks
assert submission.shape == (1659, 81)
assert submission.isna().sum().sum() == 0

# write
submission.to_csv("submission.csv", index=False)

submission.head()

,ID,PXD,Raw Data File,Characteristics[Age],Characteristics[AlkylationReagent],Characteristics[AnatomicSiteTumor],Characteristics[AncestryCategory],Characteristics[BMI],Characteristics[Bait],Characteristics[BiologicalReplicate],...,FactorValue[Bait],FactorValue[CellPart],FactorValue[Compound],FactorValue[ConcentrationOfCompound].1,FactorValue[Disease],FactorValue[FractionIdentifier],FactorValue[GeneticModification],FactorValue[Temperature],FactorValue[Treatment],Usage
0,1,PXD004010,ad_pl01.raw,Text Span,Text Span,Text Span,Text Span,Text Span,Text Span,Text Span,...,Text Span,Text Span,Text Span,Text Span,Text Span,Text Span,Text Span,Text Span,Text Span,Text Span
1,2,PXD004010,ad_pl02.raw,Text Span,Text Span,Text Span,Text Span,Text Span,Text Span,Text Span,...,Text Span,Text Span,Text Span,Text Span,Text Span,Text Span,Text Span,Text Span,Text Span,Text Span
2,3,PXD004010,ad_pl03.raw,Text Span,Text Span,Text Span,Text Span,Text Span,Text Span,Text Span,...,Text Span,Text Span,Text Span,Text Span,Text Span,Text Span,Text Span,Text Span,Text Span,Text Span
3,4,PXD004010,ad_pl04.raw,Text Span,Text Span,Text Span,Text Span,Text Span,Text Span,Text Span,...,Text Span,Text Span,Text Span,Text Span,Text Span,Text Span,Text Span,Text Span,Text Span,Text Span
4,5,PXD004010,ad_pl05.raw,Text Span,Text Span,Text Span,Text Span,Text Span,Text Span,Text Span,...,Text Span,Text Span,Text Span,Text Span,Text Span,Text Span,Text Span,Text Span,Text Span,Text Span
